# Tune multiclass need predictions

Compare network sizes, learning rates, and L2 regularization to improve the predicted need
distribution $q$ and its unrounded mean $\mu_i=\sum_h hq_i(h)$. Train fresh models on the same
training subsets and initialization seeds across candidate settings.

1. Tune settings on a small number of models **within each outer split**, using mean validation
   cross-entropy on $q$. Select each model's checkpoint using that same validation loss.
2. For each split, train the requested model family with its selected settings, reusing winning
   pilot checkpoints. Test data never select settings, checkpoints, or models.
3. Export allocation-compatible predictions and need-error diagnostics. Existing prediction files
   remain separate. This notebook does not run allocation algorithms.

Selection is separate for each split because test people in one split can appear in another split's
validation set. The cross-split leaderboard is descriptive; it is not used to choose a global winner.
Validation scores used for tuning are optimistic; held-out test results evaluate the resulting procedure.

The existing Rashomon rule is preserved: the special prediction-file loss rows still contain binary
qualification log loss on $p_i=\Pr(h_i\ge\text{threshold})$. Cross-entropy on $q$ is saved separately.


In [10]:
from copy import deepcopy
import hashlib
from itertools import product
import json
from pathlib import Path
import platform
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sklearn
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/allocations.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research repository or its test/ folder.")


## Settings

Start with `RUN_MODE = "pilot"`. Switch to `"full"` for five splits and 100 output models per split.
Pilot and full results have different identifiers and filenames. The grid contains 12 candidates.
The current architecture is (50, 100, 200); the wider architecture is (100, 200, 400).

`L2_ALPHAS` is the neural network's regularization strength; it is unrelated to allocation replacement α.
The existing input features are retained. Optional standardization is fit only on each model's sampled
training rows. Keep its setting fixed across the capacity comparison.

As in the original notebook, `BOOTSTRAP_SIZE = 0.5` selects half the training pool **without replacement**.
The same subset rule and outer split seeds are retained for matched allocation comparisons.
`NEED_BOUND = 17` fixes the existing need support 0–17; change it explicitly if the domain changes.


In [11]:
RUN_MODE = "full"  # "pilot" or "full"
PRESETS = {
    "pilot": {"n_seeds": 2, "tune_models": 3, "n_models": 5, "max_epochs": 300},
    "full": {"n_seeds": 5, "tune_models": 5, "n_models": 100, "max_epochs": 1000},
}
if RUN_MODE not in PRESETS:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'.")
N_SEEDS = PRESETS[RUN_MODE]["n_seeds"]
TUNE_MODELS = PRESETS[RUN_MODE]["tune_models"]
N_MODELS = PRESETS[RUN_MODE]["n_models"]
MAX_EPOCHS = PRESETS[RUN_MODE]["max_epochs"]

HIDDEN_LAYER_SIZES = [(50, 100, 200), (100, 200, 400)]
LEARNING_RATES = [1e-3, 3e-4]
L2_ALPHAS = [1e-4, 1e-3, 1e-2]
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.25  # Fraction of the remaining 80%: 60/20/20 overall.
BOOTSTRAP_SIZE = 0.50
CHECK_EVERY = 5
PATIENCE_CHECKS = 10
MIN_DELTA = 1e-4
STANDARDIZE_FEATURES = False
BLAS_THREADS = 1
SAVE_DISTRIBUTIONS = False
RESUME = True

METHOD = "bootstrap"
PRED_METHOD = "average"
NEED_COL = "gagne_sum_t"
NEED_BOUND = 17
NEED_THRESHOLD = 1
GROUP_COL = "race"
QUALIFICATION_COL = f"qualified_gagne_{NEED_THRESHOLD}"
RAW_DATA_PATH = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
PREDICTIONS_DIR = ROOT / "data/predictions"
NOTEBOOK_PATH = ROOT / "test/01-predictions_bootstrap_exp_needs_multiclass_finetune.ipynb"

features = [
    'dem_female', 'dem_age_band_18-24_tm1', 'dem_age_band_25-34_tm1',
    'dem_age_band_35-44_tm1', 'dem_age_band_45-54_tm1', 'dem_age_band_55-64_tm1',
    'dem_age_band_65-74_tm1', 'dem_age_band_75+_tm1', 'hypertension_elixhauser_tm1',
    'cost_dialysis_tm1', 'cost_emergency_tm1', 'cost_home_health_tm1',
    'cost_ip_medical_tm1', 'cost_ip_surgical_tm1', 'cost_laboratory_tm1',
    'cost_op_primary_care_tm1', 'cost_op_specialists_tm1', 'cost_op_surgery_tm1',
    'cost_other_tm1', 'cost_pharmacy_tm1', 'cost_physical_therapy_tm1',
    'cost_radiology_tm1', 'gagne_sum_tm1',
]


In [12]:
for name, value in {"N_SEEDS": N_SEEDS, "TUNE_MODELS": TUNE_MODELS, "N_MODELS": N_MODELS,
                    "MAX_EPOCHS": MAX_EPOCHS, "CHECK_EVERY": CHECK_EVERY,
                    "PATIENCE_CHECKS": PATIENCE_CHECKS, "BLAS_THREADS": BLAS_THREADS}.items():
    if isinstance(value, bool) or not isinstance(value, int) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")
if TUNE_MODELS > N_MODELS or not 0 < BOOTSTRAP_SIZE <= 1:
    raise ValueError("Require TUNE_MODELS <= N_MODELS and 0 < BOOTSTRAP_SIZE <= 1.")
if not 0 < TEST_SIZE < 1 or not 0 < VALIDATION_SIZE < 1 or MIN_DELTA < 0:
    raise ValueError("Invalid split fractions or MIN_DELTA.")
if not isinstance(NEED_BOUND, int) or NEED_BOUND < 1 or not 1 <= NEED_THRESHOLD <= NEED_BOUND:
    raise ValueError("Specify a positive integer need bound and a threshold within that support.")
if not HIDDEN_LAYER_SIZES or not LEARNING_RATES or not L2_ALPHAS:
    raise ValueError("The tuning grid cannot be empty.")
for layers in HIDDEN_LAYER_SIZES:
    if not layers or any(isinstance(n, bool) or not isinstance(n, int) or n < 1 for n in layers):
        raise ValueError("Every hidden layer size must be a positive integer.")
if any(not np.isfinite(v) or v <= 0 for v in LEARNING_RATES):
    raise ValueError("Learning rates must be finite and positive.")
if any(not np.isfinite(v) or v < 0 for v in L2_ALPHAS):
    raise ValueError("L2 strengths must be finite and nonnegative.")

df = pd.read_csv(RAW_DATA_PATH)
required = ["person_id", "cost_t", NEED_COL, GROUP_COL, QUALIFICATION_COL, *features]
missing = [name for name in required if name not in df]
if missing:
    raise ValueError(f"Missing columns: {missing}")
if not df.person_id.is_unique or df.person_id.isna().any() or df.person_id.lt(0).any():
    raise ValueError("Person IDs must be unique, nonmissing, and nonnegative.")
X = df[features].to_numpy(dtype=float)
raw_h = df[NEED_COL].to_numpy(dtype=float)
if not np.isfinite(X).all() or not np.isfinite(raw_h).all():
    raise ValueError("Features and true needs must be finite.")
if (raw_h != np.floor(raw_h)).any() or (raw_h < 0).any() or (raw_h > NEED_BOUND).any():
    raise ValueError("True needs must be integers within the configured need support.")
h = raw_h.astype(int)
y = df[QUALIFICATION_COL].to_numpy()
if not np.isin(y, [0, 1]).all() or not np.array_equal(y, (h >= NEED_THRESHOLD).astype(int)):
    raise ValueError("Qualification labels must match the configured need threshold.")
if not df[GROUP_COL].isin([0, 1]).all():
    raise ValueError("Group labels must be 0 or 1.")
NEED_CLASSES = np.arange(NEED_BOUND + 1)

def split_indices(seed):
    train, test = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=seed)
    train, validation = train_test_split(train, test_size=VALIDATION_SIZE, random_state=seed)
    return {"train": train, "validation": validation, "test": test}

def training_subset(train_indices, model_index):
    # Match np.random.seed(i); np.random.permutation(...) from the original runner,
    # without changing NumPy's global random state.
    count = int(len(train_indices) * BOOTSTRAP_SIZE)
    if count < 1:
        raise ValueError("The training subset is empty.")
    positions = np.random.RandomState(model_index).permutation(len(train_indices))[:count]
    return train_indices[positions]

candidates = []
for layers, learning_rate, l2 in product(HIDDEN_LAYER_SIZES, LEARNING_RATES, L2_ALPHAS):
    settings = {"hidden_layer_sizes": list(layers), "learning_rate_init": learning_rate, "l2_alpha": l2}
    candidate_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
    candidates.append({"candidate_id": candidate_id, **settings})
if len({c["candidate_id"] for c in candidates}) != len(candidates):
    raise ValueError("Duplicate tuning candidates.")
candidate_by_id = {c["candidate_id"]: c for c in candidates}
splits = {seed: split_indices(seed) for seed in range(N_SEEDS)}
print(f"{len(candidates)} settings × {TUNE_MODELS} pilot models × {N_SEEDS} splits = {len(candidates)*TUNE_MODELS*N_SEEDS} tuning fits")
print(f"Up to {(N_MODELS-TUNE_MODELS)*N_SEEDS} additional fits for the selected families; completed fits are resumable.")
display(pd.DataFrame(candidates))


12 settings × 5 pilot models × 5 splits = 300 tuning fits
Up to 475 additional fits for the selected families; completed fits are resumable.


,candidate_id,hidden_layer_sizes,learning_rate_init,l2_alpha
0,0ff9aa093ac4,"[50, 100, 200]",0.0010,0.0001
1,d7d7c475284d,"[50, 100, 200]",0.0010,0.0010
2,eca9249f4377,"[50, 100, 200]",0.0010,0.0100
3,36a6187b9c23,"[50, 100, 200]",0.0003,0.0001
4,f825b49fddd2,"[50, 100, 200]",0.0003,0.0010
5,81b5e88c238e,"[50, 100, 200]",0.0003,0.0100
6,38502c736bfe,"[100, 200, 400]",0.0010,0.0001
7,784b3cd44641,"[100, 200, 400]",0.0010,0.0010
8,8acfd48d0ad3,"[100, 200, 400]",0.0010,0.0100
9,b62a376ce481,"[100, 200, 400]",0.0003,0.0001


## Validation checkpoints and diagnostics

Training uses one epoch at a time through the public `MLPClassifier.partial_fit` API, supplying
the complete class support even when a training subset is missing a need class. Each candidate
receives the same subset and epoch shuffle order for a given model seed.

The lowest observed validation cross-entropy checkpoint is retained. Training stops after
`PATIENCE_CHECKS` checks without an improvement of at least `MIN_DELTA` (checks occur every
`CHECK_EVERY` epochs, also at the first and last epoch). The best checkpoint can precede the stopping
epoch. This uses cross-entropy directly; sklearn's built-in classification early stopping monitors
accuracy. See the [MLPClassifier documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

Diagnostics include cross-entropy on q, qualification log loss on p, unrounded expected-need MAE/RMSE,
and signed mean prediction error (predicted minus true need) within each group. Positive bias means
overprediction. `bias_gap` is group-1 bias minus group-0 bias. These are prediction errors, not allocation disparity.
Actual training-subset metrics and full training-pool metrics are reported separately.


In [13]:
def derive_predictions(model, values, scaler=None):
    if not np.array_equal(model.classes_, NEED_CLASSES):
        raise ValueError("Unexpected model class support or ordering.")
    transformed = values if scaler is None else scaler.transform(values)
    q = model.predict_proba(transformed)
    if q.shape != (len(values), len(NEED_CLASSES)) or not np.isfinite(q).all():
        raise ValueError("Invalid multiclass predictions.")
    if (q < 0).any() or not np.allclose(q.sum(axis=1), 1.0, atol=1e-7):
        raise ValueError("Predicted need probabilities do not sum to one.")
    p = np.clip(q[:, NEED_CLASSES >= NEED_THRESHOLD].sum(axis=1), 0.0, 1.0)
    mu = q @ NEED_CLASSES.astype(float)
    return q, p, mu

def prediction_metrics(indices, q, p, mu, prefix):
    error = mu - h[indices]
    labels = df.iloc[indices][GROUP_COL].to_numpy()
    metrics = {
        "cross_entropy": float(log_loss(h[indices], q, labels=NEED_CLASSES)),
        "qualification_log_loss": float(log_loss(y[indices], p, labels=[0, 1])),
        "need_mae": float(np.mean(np.abs(error))),
        "need_rmse": float(np.sqrt(np.mean(error**2))),
        "need_bias": float(np.mean(error)),
    }
    for group in (0, 1):
        group_error = error[labels == group]
        metrics[f"group_{group}_n"] = len(group_error)
        metrics[f"group_{group}_bias"] = float(group_error.mean()) if len(group_error) else None
        metrics[f"group_{group}_mae"] = float(np.abs(group_error).mean()) if len(group_error) else None
        metrics[f"group_{group}_rmse"] = float(np.sqrt(np.mean(group_error**2))) if len(group_error) else None
    b0, b1 = metrics["group_0_bias"], metrics["group_1_bias"]
    metrics["bias_gap"] = None if b0 is None or b1 is None else b1 - b0
    return {f"{prefix}_{name}": value for name, value in metrics.items()}

def fit_with_validation(split, candidate, model_index):
    # This function receives training and validation indices only.
    train_ids = training_subset(split["train"], model_index)
    validation_ids = split["validation"]
    scaler = StandardScaler().fit(X[train_ids]) if STANDARDIZE_FEATURES else None
    train_x = X[train_ids] if scaler is None else scaler.transform(X[train_ids])
    val_x = X[validation_ids] if scaler is None else scaler.transform(X[validation_ids])
    model = MLPClassifier(
        hidden_layer_sizes=tuple(candidate["hidden_layer_sizes"]), solver="adam", activation="relu",
        alpha=candidate["l2_alpha"], learning_rate_init=candidate["learning_rate_init"],
        batch_size="auto", shuffle=False, random_state=model_index,
        early_stopping=False, max_iter=1, tol=0.0, n_iter_no_change=MAX_EPOCHS + 1,
    )
    shuffle_rng = np.random.RandomState(model_index + 100000)
    best_model, best_loss, patience_loss = None, np.inf, np.inf
    best_epoch, stale_checks, history = 0, 0, []
    with threadpool_limits(limits=BLAS_THREADS):
        for epoch in range(1, MAX_EPOCHS + 1):
            order = shuffle_rng.permutation(len(train_ids))
            model.partial_fit(train_x[order], h[train_ids][order], classes=NEED_CLASSES)
            if epoch != 1 and epoch % CHECK_EVERY != 0 and epoch != MAX_EPOCHS:
                continue
            q_val = model.predict_proba(val_x)
            value = float(log_loss(h[validation_ids], q_val, labels=NEED_CLASSES))
            if not np.isfinite(value):
                raise ValueError("Nonfinite validation loss; inspect this candidate's training settings.")
            improved = value < best_loss
            if improved:
                best_model, best_loss, best_epoch = deepcopy(model), value, epoch
            if value < patience_loss - MIN_DELTA:
                patience_loss, stale_checks = value, 0
            else:
                stale_checks += 1
            history.append({"epoch": epoch, "training_optimizer_loss": float(model.loss_),
                            "validation_cross_entropy": value, "best_so_far": best_loss,
                            "new_best": bool(improved)})
            if stale_checks >= PATIENCE_CHECKS:
                break
        if best_model is None:
            raise RuntimeError("No validation checkpoint was produced.")
        metrics = {"best_epoch": best_epoch, "epochs_trained": epoch,
                   "stopped_early": epoch < MAX_EPOCHS, "hit_epoch_limit": epoch == MAX_EPOCHS,
                   "n_train_sample": len(train_ids)}
        for prefix, indices in (("training_subset", train_ids), ("validation", validation_ids)):
            metrics.update(prediction_metrics(indices, *derive_predictions(best_model, X[indices], scaler), prefix))
    if not np.isclose(metrics["validation_cross_entropy"], best_loss, rtol=0, atol=1e-12):
        raise AssertionError("The retained checkpoint does not reproduce its best validation loss.")
    return {"model": best_model, "scaler": scaler}, metrics, pd.DataFrame(history)


## Resumable runs

Save this notebook before running. A run identifier includes the saved code, settings, input hash,
and library versions. Checkpoints, learning curves, split identities, and validation summaries live
under `data/predictions/finetune/`. Completed fits are reused only after their metadata and hashes match.
An interrupted fit is restarted, while completed fits are retained. Failed candidates stop the run;
they are not silently excluded from the comparison.


In [14]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

source = json.loads(NOTEBOOK_PATH.read_text())
code_text = "\n\n".join("".join(cell["source"]) for cell in source["cells"] if cell["cell_type"] == "code")
config = {
    "schema_version": 1, "run_mode": RUN_MODE, "n_seeds": N_SEEDS,
    "n_models": N_MODELS, "tune_models": TUNE_MODELS, "candidates": candidates,
    "test_size": TEST_SIZE, "validation_size": VALIDATION_SIZE, "bootstrap_size": BOOTSTRAP_SIZE,
    "max_epochs": MAX_EPOCHS, "check_every": CHECK_EVERY, "patience_checks": PATIENCE_CHECKS,
    "min_delta": MIN_DELTA, "standardize_features": STANDARDIZE_FEATURES, "blas_threads": BLAS_THREADS,
    "save_distributions": SAVE_DISTRIBUTIONS, "features": features, "need_column": NEED_COL,
    "group_column": GROUP_COL, "need_bound": NEED_BOUND, "need_threshold": NEED_THRESHOLD,
    "method": METHOD, "prediction_method": PRED_METHOD,
    "selection_rule": "lowest_mean_validation_q_cross_entropy_within_each_split",
    "raw_data_path": str(RAW_DATA_PATH), "raw_data_sha256": file_hash(RAW_DATA_PATH),
    "code_sha256": hashlib.sha256(code_text.encode()).hexdigest(),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
}
config_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
MODEL_CLASS = f"nn_multiclass_finetune_{RUN_MODE}_{config_id}"
RUN_DIR = PREDICTIONS_DIR / "finetune" / MODEL_CLASS
RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / "config.json"
if config_path.exists():
    if json.loads(config_path.read_text()) != config:
        raise ValueError("A different configuration already exists in this run directory.")
else:
    atomic_json(config_path, config)
split_frames = [pd.DataFrame({"seed": seed, "split": name, "person_id": df.iloc[indices].person_id.to_numpy()})
                for seed, split in splits.items() for name, indices in split.items()]
atomic_csv(RUN_DIR / "split_people.csv", pd.concat(split_frames, ignore_index=True))

def trial_paths(seed, candidate_id, model_index):
    folder = RUN_DIR / "trials" / f"seed-{seed}" / candidate_id / f"m_{model_index + 1}"
    return {"checkpoint": folder / "checkpoint.joblib", "history": folder / "history.csv", "metadata": folder / "trial.json"}

def read_completed_trial(seed, candidate, model_index):
    paths = trial_paths(seed, candidate["candidate_id"], model_index)
    if not paths["metadata"].exists():
        return None
    metadata = json.loads(paths["metadata"].read_text())
    expected = {"config_id": config_id, "seed": seed, "candidate_id": candidate["candidate_id"], "model_index": model_index}
    if any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError(f"Mismatched trial metadata: {paths['metadata']}")
    for name in ("checkpoint", "history"):
        if not paths[name].exists() or file_hash(paths[name]) != metadata["sha256"][name]:
            raise ValueError(f"Incomplete or changed checkpoint: {paths[name]}")
    return metadata

def run_trial(seed, candidate, model_index):
    completed = read_completed_trial(seed, candidate, model_index) if RESUME else None
    if completed is not None:
        return completed
    paths = trial_paths(seed, candidate["candidate_id"], model_index)
    paths["metadata"].parent.mkdir(parents=True, exist_ok=True)
    started = perf_counter()
    # No test arrays or test labels are passed to training/checkpoint selection.
    tuning_split = {name: splits[seed][name] for name in ("train", "validation")}
    bundle, metrics, history = fit_with_validation(tuning_split, candidate, model_index)
    elapsed = perf_counter() - started
    paths["metadata"].unlink(missing_ok=True)
    temporary = paths["checkpoint"].with_suffix(".joblib.tmp")
    joblib.dump(bundle, temporary, compress=3)
    temporary.replace(paths["checkpoint"])
    atomic_csv(paths["history"], history)
    metadata = {"config_id": config_id, "seed": seed, "candidate_id": candidate["candidate_id"],
                "model_index": model_index, "model": f"m_{model_index + 1}",
                "elapsed_seconds": elapsed, **metrics,
                "sha256": {name: file_hash(paths[name]) for name in ("checkpoint", "history")}}
    atomic_json(paths["metadata"], metadata)
    return metadata

print("Run directory:", RUN_DIR)
print("Allocation MODEL_CLASS:", MODEL_CLASS)


Run directory: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/predictions/finetune/nn_multiclass_finetune_full_76aad3c8aee4
Allocation MODEL_CLASS: nn_multiclass_finetune_full_76aad3c8aee4


## Tune on validation data

Every candidate must complete the same pilot model seeds. The winner for each outer split minimizes
mean validation cross-entropy; ties use the stable candidate ID. MAE, RMSE, group bias, and variability
are shown for interpretation, but they do not silently change the selection criterion.
The pooled candidate summary is descriptive because the split evaluations overlap.


In [15]:
tuning_rows = []
for seed in range(N_SEEDS):
    for candidate in tqdm(candidates, desc=f"Tune split {seed}"):
        for model_index in range(TUNE_MODELS):
            metadata = run_trial(seed, candidate, model_index)
            tuning_rows.append({key: value for key, value in metadata.items() if key != "sha256"})
tuning_results = pd.DataFrame(tuning_rows)
expected_keys = {(seed, c["candidate_id"], i) for seed in range(N_SEEDS)
                 for c in candidates for i in range(TUNE_MODELS)}
actual_keys = list(zip(tuning_results.seed, tuning_results.candidate_id, tuning_results.model_index))
if set(actual_keys) != expected_keys or len(actual_keys) != len(expected_keys):
    raise ValueError("Incomplete or duplicate tuning coverage.")
if not np.isfinite(tuning_results.validation_cross_entropy).all():
    raise ValueError("All candidates must have finite validation cross-entropy.")

def choose_settings(results):
    # Never read any test metric when choosing configurations.
    scores = results.groupby(["seed", "candidate_id"]).agg(
        mean_validation_cross_entropy=("validation_cross_entropy", "mean"),
        std_validation_cross_entropy=("validation_cross_entropy", "std"),
        validation_need_mae=("validation_need_mae", "mean"),
        validation_need_rmse=("validation_need_rmse", "mean"),
        validation_group_0_bias=("validation_group_0_bias", "mean"),
        validation_group_1_bias=("validation_group_1_bias", "mean"),
        validation_bias_gap=("validation_bias_gap", "mean"),
        mean_best_epoch=("best_epoch", "mean"), n_models=("model_index", "size"),
    ).reset_index()
    winners = (scores.sort_values(["seed", "mean_validation_cross_entropy", "candidate_id"])
               .groupby("seed", as_index=False).head(1).reset_index(drop=True))
    return scores, winners

validation_scores, selected_settings = choose_settings(tuning_results)
candidate_details = pd.DataFrame(candidates)
selected_settings = selected_settings.merge(candidate_details, on="candidate_id", validate="many_to_one")
selected_by_seed = {int(row.seed): candidate_by_id[row.candidate_id] for row in selected_settings.itertuples()}
validation_leaderboard = validation_scores.groupby("candidate_id").agg(
    mean_validation_cross_entropy=("mean_validation_cross_entropy", "mean"),
    validation_need_mae=("validation_need_mae", "mean"),
    validation_need_rmse=("validation_need_rmse", "mean"),
).reset_index().merge(candidate_details, on="candidate_id", validate="one_to_one").sort_values("mean_validation_cross_entropy")
atomic_csv(RUN_DIR / "tuning_results.csv", tuning_results)
atomic_csv(RUN_DIR / "validation_scores_by_seed.csv", validation_scores)
atomic_csv(RUN_DIR / "validation_leaderboard.csv", validation_leaderboard)
atomic_csv(RUN_DIR / "selected_settings.csv", selected_settings)
display(selected_settings)
display(validation_leaderboard)


Tune split 4: 100%|██████████| 12/12 [08:35<00:00, 42.99s/it]


,seed,candidate_id,mean_validation_cross_entropy,std_validation_cross_entropy,validation_need_mae,validation_need_rmse,validation_group_0_bias,validation_group_1_bias,validation_bias_gap,mean_best_epoch,n_models,hidden_layer_sizes,learning_rate_init,l2_alpha
0,0,81b5e88c238e,0.897472,0.002798,0.562392,1.030975,0.022878,-0.057969,-0.080847,192.0,5,"[50, 100, 200]",0.0003,0.01
1,1,81b5e88c238e,0.899713,0.001977,0.571914,1.063872,0.002827,-0.021906,-0.024733,222.0,5,"[50, 100, 200]",0.0003,0.01
2,2,81b5e88c238e,0.888564,0.004453,0.556576,1.022619,0.028291,-0.121326,-0.149616,185.0,5,"[50, 100, 200]",0.0003,0.01
3,3,81b5e88c238e,0.906044,0.003136,0.582836,1.074891,0.008291,-0.118971,-0.127261,198.0,5,"[50, 100, 200]",0.0003,0.01
4,4,81b5e88c238e,0.883273,0.002157,0.556410,1.024561,0.011630,-0.101254,-0.112884,167.0,5,"[50, 100, 200]",0.0003,0.01


,candidate_id,mean_validation_cross_entropy,validation_need_mae,validation_need_rmse,hidden_layer_sizes,learning_rate_init,l2_alpha
4,81b5e88c238e,0.895013,0.566026,1.043384,"[50, 100, 200]",0.0003,0.0100
7,9bdbe152f5b5,0.896786,0.565117,1.043484,"[100, 200, 400]",0.0003,0.0100
10,eca9249f4377,0.897371,0.567460,1.044858,"[50, 100, 200]",0.0010,0.0100
11,f825b49fddd2,0.898628,0.566574,1.044917,"[50, 100, 200]",0.0003,0.0010
5,8acfd48d0ad3,0.899160,0.567952,1.045907,"[100, 200, 400]",0.0010,0.0100
1,36a6187b9c23,0.900147,0.566762,1.044610,"[50, 100, 200]",0.0003,0.0001
6,967f80ad4a43,0.900479,0.565875,1.044612,"[100, 200, 400]",0.0003,0.0010
9,d7d7c475284d,0.900696,0.564605,1.046421,"[50, 100, 200]",0.0010,0.0010
8,b62a376ce481,0.901371,0.564308,1.045951,"[100, 200, 400]",0.0003,0.0001
0,0ff9aa093ac4,0.901807,0.567935,1.046564,"[50, 100, 200]",0.0010,0.0001


## Inspect validation learning curves

The curve for model m_1 of each selected split illustrates checkpoint selection. All model/candidate
histories are saved separately. Large between-model variation in the tables is a reason to inspect
more histories before interpreting a small improvement in the mean.


In [16]:
learning_curve = go.Figure()
for seed, candidate in selected_by_seed.items():
    paths = trial_paths(seed, candidate["candidate_id"], 0)
    history = pd.read_csv(paths["history"])
    metadata = read_completed_trial(seed, candidate, 0)
    learning_curve.add_trace(go.Scatter(x=history.epoch, y=history.validation_cross_entropy,
                                        mode="lines", name=f"Split {seed}, m_1"))
    learning_curve.add_trace(go.Scatter(x=[metadata["best_epoch"]], y=[metadata["validation_cross_entropy"]],
                                        mode="markers", marker=dict(size=9, symbol="diamond"),
                                        name=f"Split {seed} checkpoint", showlegend=False))
learning_curve.update_layout(template="plotly_white", title="Selected models: validation cross-entropy",
                              xaxis_title="Training epoch", yaxis_title="Cross-entropy on q")
learning_curve.show()
learning_curve.write_html(RUN_DIR / "validation_learning_curves.html", include_plotlyjs=True)


## Produce the selected prediction families

The settings are now fixed separately for each split. Reuse the winning pilot models and fit any
remaining model seeds. Predictions use each model's best validation checkpoint; training is not
repeated on train+validation, so the validation loss rows retain their original meaning.

Test diagnostics are calculated here, after selection. The final prediction CSV has the same
`m_i`, `exp_need_m_i`, person metadata, and -2/-1 training/validation loss rows as the existing runner.
Those two loss rows contain **qualification log loss on p**. The separate loss file contains both
q-based cross-entropy and explicitly named qualification losses, plus expected-need error metrics.


In [17]:
output_frames, final_loss_rows = [], []
for seed in range(N_SEEDS):
    split = splits[seed]
    candidate = selected_by_seed[seed]
    predictions, expected_needs, training_loss, validation_loss = {}, {}, {}, {}
    for model_index in tqdm(range(N_MODELS), desc=f"Selected family, split {seed}"):
        # Reuse winning tuning checkpoints even if RESUME=False requested a fresh tuning pass.
        metadata = read_completed_trial(seed, candidate, model_index) if model_index < TUNE_MODELS else None
        if metadata is None:
            metadata = run_trial(seed, candidate, model_index)
        paths = trial_paths(seed, candidate["candidate_id"], model_index)
        # Only locally generated, hash-verified checkpoints from this configuration are loaded.
        verified = read_completed_trial(seed, candidate, model_index)
        bundle = joblib.load(paths["checkpoint"])
        model, scaler = bundle["model"], bundle["scaler"]
        name = f"m_{model_index + 1}"
        record = {key: value for key, value in verified.items() if key != "sha256"}
        record.update({"n_iter": verified["epochs_trained"],
                       "hidden_layer_sizes": candidate["hidden_layer_sizes"],
                       "learning_rate_init": candidate["learning_rate_init"], "l2_alpha": candidate["l2_alpha"]})
        with threadpool_limits(limits=BLAS_THREADS):
            for prefix, indices in (("training", split["train"]), ("validation", split["validation"]), ("test", split["test"])):
                q, p, mu = derive_predictions(model, X[indices], scaler)
                record.update(prediction_metrics(indices, q, p, mu, prefix))
                if prefix == "test":
                    predictions[name], expected_needs[f"exp_need_{name}"] = p, mu
                    if SAVE_DISTRIBUTIONS:
                        folder = RUN_DIR / "distributions"
                        folder.mkdir(exist_ok=True)
                        np.savez_compressed(folder / f"seed-{seed}_{name}.npz",
                                            person_id=df.iloc[indices].person_id.to_numpy(), classes=NEED_CLASSES, q=q)
        training_loss[name] = record["training_qualification_log_loss"]
        validation_loss[name] = record["validation_qualification_log_loss"]
        final_loss_rows.append(record)

    test_frame = df.iloc[split["test"]]
    frame = pd.concat([pd.DataFrame(predictions), pd.DataFrame([training_loss, validation_loss])], ignore_index=True)
    frame["person_id"] = np.concatenate([test_frame.person_id.to_numpy(), [-2, -1]])
    frame["y"] = np.concatenate([y[split["test"]], [np.nan, np.nan]])
    frame["cost_t"] = np.concatenate([test_frame.cost_t.to_numpy(), [np.nan, np.nan]])
    frame[NEED_COL] = np.concatenate([h[split["test"]], [np.nan, np.nan]])
    frame["seed"] = seed
    frame = pd.concat([frame, pd.DataFrame(expected_needs).reindex(frame.index)], axis=1)
    output_frames.append(frame)

df_output = pd.concat(output_frames, ignore_index=True)
for column in df_output:
    if column.startswith("m_"):
        df_output[column] = df_output[column].astype("float32")
df_nn_losses = pd.DataFrame(final_loss_rows)
expected_final = {(seed, f"m_{i+1}") for seed in range(N_SEEDS) for i in range(N_MODELS)}
if set(zip(df_nn_losses.seed, df_nn_losses.model)) != expected_final or len(df_nn_losses) != len(expected_final):
    raise ValueError("Final model coverage is incomplete or duplicated.")
display(df_nn_losses[["seed", "model", "best_epoch", "epochs_trained", "training_subset_cross_entropy",
                      "validation_cross_entropy", "test_cross_entropy", "test_need_mae", "test_need_rmse",
                      "test_group_0_bias", "test_group_1_bias", "test_bias_gap"]])


Selected family, split 4: 100%|██████████| 100/100 [16:07<00:00,  9.67s/it]


,seed,model,best_epoch,epochs_trained,training_subset_cross_entropy,validation_cross_entropy,test_cross_entropy,test_need_mae,test_need_rmse,test_group_0_bias,test_group_1_bias,test_bias_gap
0,0,m_1,150,200,0.861279,0.901885,0.889476,0.562475,1.034449,0.034862,-0.071928,-0.106790
1,0,m_2,265,315,0.822355,0.896016,0.885863,0.555367,1.037641,0.006075,-0.119769,-0.125844
2,0,m_3,155,205,0.860582,0.896831,0.882980,0.557688,1.035858,0.042060,-0.049271,-0.091331
3,0,m_4,215,265,0.842297,0.898136,0.886668,0.561894,1.036706,0.027302,-0.071032,-0.098334
4,0,m_5,175,225,0.850860,0.894492,0.880700,0.555858,1.031212,0.013694,-0.104629,-0.118323
...,...,...,...,...,...,...,...,...,...,...,...,...
495,4,m_96,185,235,0.859184,0.879094,0.865520,0.546530,0.999707,0.005893,-0.147083,-0.152976
496,4,m_97,140,190,0.881150,0.887301,0.872267,0.548825,0.998972,0.029499,-0.114183,-0.143682
497,4,m_98,170,220,0.850131,0.881172,0.871008,0.554877,1.001399,0.034468,-0.119097,-0.153565
498,4,m_99,240,290,0.835614,0.879920,0.862629,0.542785,1.002457,0.014029,-0.125590,-0.139619


In [18]:
summary_metrics = ["validation_cross_entropy", "test_cross_entropy", "test_qualification_log_loss",
                   "test_need_mae", "test_need_rmse", "test_group_0_bias", "test_group_1_bias", "test_bias_gap"]
prediction_summary = df_nn_losses.groupby("seed")[summary_metrics].agg(["mean", "std"])
display(prediction_summary)
print("SD describes variation across models within each split; it is not a confidence interval.")

tag = f"test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_{QUALIFICATION_COL}"
fp_output = PREDICTIONS_DIR / f"{tag}.csv"
fp_nn_losses = PREDICTIONS_DIR / "losses" / f"{tag}.csv"
export_marker = RUN_DIR / "exports.json"
if export_marker.exists():
    previous = json.loads(export_marker.read_text())
    if (previous.get("config_id") != config_id or previous.get("predictions") != str(fp_output)
            or previous.get("losses") != str(fp_nn_losses)):
        raise ValueError("Existing exports belong to a different configuration.")
    if previous.get("status") == "complete":
        for name, path in (("predictions", fp_output), ("losses", fp_nn_losses)):
            if not path.exists() or file_hash(path) != previous["sha256"][name]:
                raise ValueError(f"A previous export was changed: {path}")
    elif previous.get("status") != "writing":
        raise ValueError("Unknown export completion state.")
elif fp_output.exists() or fp_nn_losses.exists():
    raise FileExistsError("Export paths already exist without matching completion metadata; inspect them before replacing.")
atomic_json(export_marker, {"config_id": config_id, "status": "writing",
                           "predictions": str(fp_output), "losses": str(fp_nn_losses)})
atomic_csv(fp_output, df_output)
atomic_csv(fp_nn_losses, df_nn_losses)
flat_summary = prediction_summary.copy()
flat_summary.columns = [f"{metric}_{stat}" for metric, stat in flat_summary.columns]
atomic_csv(RUN_DIR / "prediction_summary.csv", flat_summary.reset_index())
atomic_json(export_marker, {"config_id": config_id, "status": "complete", "model_class": MODEL_CLASS,
                           "predictions": str(fp_output), "losses": str(fp_nn_losses),
                           "n_seeds": N_SEEDS, "n_models": N_MODELS,
                           "selected_candidates_by_seed": {str(seed): c["candidate_id"] for seed, c in selected_by_seed.items()},
                           "sha256": {"predictions": file_hash(fp_output), "losses": file_hash(fp_nn_losses)}})
print("Saved predictions:", fp_output)
print("Saved losses and diagnostics:", fp_nn_losses)
print("For the allocation notebook, use:")
print(f'MODEL_CLASS = "{MODEL_CLASS}"')
print(f'N_MODELS = {N_MODELS}')
print('PRED_METHOD = "average"')


validation_cross_entropy           test_cross_entropy            \
                         mean       std               mean       std   
seed                                                                   
0                    0.896506  0.002930           0.884311  0.002957   
1                    0.899807  0.003000           0.898761  0.002451   
2                    0.887306  0.002717           0.887284  0.002606   
3                    0.906766  0.002787           0.884494  0.002805   
4                    0.883597  0.002259           0.868219  0.002574   

     test_qualification_log_loss           test_need_mae            \
                            mean       std          mean       std   
seed                                                                 
0                       0.323752  0.001057      0.559763  0.004797   
1                       0.327896  0.001047      0.560911  0.004526   
2                       0.323524  0.000929      0.559824  0.004201   
3                       0.330863  0.000855      0.556618  0.003938   
4                       0.320537  0.000932      0.547992  0.004184   

     test_need_rmse           test_group_0_bias           test_group_1_bias  \
               mean       std              mean       std              mean   
seed                                                                          
0          1.036870  0.002999          0.014466  0.014906         -0.098130   
1          1.027111  0.002866          0.012631  0.015068         -0.081985   
2          1.029704  0.002884          0.021496  0.014455         -0.083799   
3          1.036768  0.002622          0.034479  0.012768         -0.094028   
4          1.001942  0.002030          0.024864  0.012881         -0.121942   

               test_bias_gap            
           std          mean       std  
seed                                    
0     0.023647     -0.112596  0.011861  
1     0.020464     -0.094616  0.010159  
2     0.020534     -0.105295  0.009973  
3     0.020214     -0.128506  0.010514  
4     0.017992     -0.146805  0.009264

SD describes variation across models within each split; it is not a confidence interval.
Saved predictions: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/predictions/test_bootstrap_nn_multiclass_finetune_full_76aad3c8aee4_average_qualified_gagne_1.csv
Saved losses and diagnostics: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/predictions/losses/test_bootstrap_nn_multiclass_finetune_full_76aad3c8aee4_average_qualified_gagne_1.csv
For the allocation notebook, use:
MODEL_CLASS = "nn_multiclass_finetune_full_76aad3c8aee4"
N_MODELS = 100
PRED_METHOD = "average"


## Allocation follow-up

Use the printed `MODEL_CLASS`, `N_MODELS`, and `PRED_METHOD` in the allocation runner. Keep the
selection rate, sample size, number of samples per seed, utility tolerance, and Rashomon tolerance
fixed when comparing against the existing prediction family. Run LDA-exp, LDA-tru, and OPT as before;
evaluate every allocation using true needs. No rounding is applied to exported expected needs here.

For the final comparison, use a full run: the pilot has fewer splits and candidate models, which
also changes the candidate Rashomon pool. Better validation cross-entropy or need RMSE alone does
not establish lower allocation disparity; check the downstream allocation results separately.
